# CNN «Некорректная укладка» бедра (`hip_positioning`)

Обучение — только здесь, в Colab (T4). Локально в репозитории ничего не
обучается, только читаются готовые веса. Цель — проверить, есть ли в
пикселях сигнал, который не ловят rule-based геометрические ориентиры.

Данные — `hip_positioning_dataset.zip`, собранный `uv run python -m columba.hip_export`
(150 изображений бедра, уже приведены к изотропному пикселю и одному виду:
правое бедро отражено к виду левого). `labels.csv` внутри архива без
UID/тегов DICOM — только обезличенный `study_folder` (`sNNN`).

**Бюджет:** ≤15 минут на T4 (Runtime -> Change runtime type -> GPU).

**Как запустить:**
1. Залить `hip_positioning_dataset.zip` в свой Google Drive.
2. Указать путь к нему в переменной `DATASET_ZIP_DRIVE_PATH` в следующей ячейке.
3. Runtime -> Run all.
4. Результаты (`oof_train.csv`, `val_pred.csv`, `hip_positioning_cnn.pt`, `report.json`)
   сохранятся в `OUTPUT_DIR_DRIVE_PATH` на Drive — их нужно скачать в
   `artifacts/models/` (веса) и `artifacts/` (отчёты/предсказания) репозитория.

In [ ]:
# --- Пути на Google Drive: единственное, что нужно поменять ---
DATASET_ZIP_DRIVE_PATH = "/content/drive/MyDrive/columba/hip_positioning_dataset.zip"
OUTPUT_DIR_DRIVE_PATH = "/content/drive/MyDrive/columba/hip_positioning_cnn_output"

SEED = 42
TIME_BUDGET_MIN = 15.0

## 1. Монтирование Drive и распаковка датасета

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(OUTPUT_DIR_DRIVE_PATH, exist_ok=True)

In [ ]:
import zipfile
from pathlib import Path

DATA_DIR = Path("/content/hip_positioning_dataset")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(DATASET_ZIP_DRIVE_PATH) as archive:
    archive.extractall(DATA_DIR)

IMAGES_DIR = DATA_DIR / "images"
LABELS_CSV = DATA_DIR / "labels.csv"
assert IMAGES_DIR.exists() and LABELS_CSV.exists()
print("PNG:", len(list(IMAGES_DIR.glob('*.png'))))

## 2. Зависимости

В Colab уже есть `torch`, `torchvision`, `pandas`, `numpy`, `scikit-learn`,
`Pillow` — ставить нечего. Ячейка ниже — на случай окружения без них.

In [ ]:
import importlib
missing = [pkg for pkg in ("torch", "torchvision", "sklearn") if importlib.util.find_spec(pkg) is None]
if missing:
    import sys
    !{sys.executable} -m pip install -q {' '.join(missing)}
print("нет в окружении и было установлено:", missing or "ничего, всё уже есть")

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

T_START = time.time()

## 3. Таблица меток

In [ ]:
labels = pd.read_csv(LABELS_CSV)
labels["png_path"] = labels["dedup_group_id"].apply(lambda gid: str(IMAGES_DIR / f"{gid}.png"))

train_df = labels[labels["split"] == "train"].reset_index(drop=True)
val_df = labels[labels["split"] == "val"].reset_index(drop=True)

print("train:", len(train_df), "позитивов:", int(train_df["label_hip_positioning"].sum()))
print("val:", len(val_df), "позитивов:", int(val_df["label_hip_positioning"].sum()))

## 4. Датасет и аугментации

PNG уже в изотропном пространстве и в едином виде (правое бедро отражено —
см. `hip_export.preprocess_for_cnn`), поэтому здесь только вписывание в
квадрат (letterbox) и аугментации:
* яркость/контраст;
* малые повороты ±5° (кадр уже изотропный, поворот честный);
* БЕЗ горизонтального флипа — стороны уже приведены к одному виду;
* маскирование углов случайными прямоугольниками — чтобы сеть не опиралась
  на реальные чёрные прямоугольники постобработки (`has_mask_rect`), которые
  сами по себе не признак нарушения (маскирование мягких тканей
  лаборантом при постобработке, см. `stages/stage_0.md`).

In [ ]:
INPUT_SIZE = 224  # letterbox-квадрат; тот же порядок, что region_cnn/spine_objects_cnn
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

AUG_MAX_ROTATION_DEG = 5.0
AUG_BRIGHTNESS = 0.10
AUG_CONTRAST = 0.10
AUG_MASK_PROB = 0.5          # вероятность маскирования каждого угла отдельно
AUG_MASK_MAX_FRACTION = 0.22  # максимальная доля высоты/ширины под маской угла


def load_png_float(path: str) -> np.ndarray:
    array = np.asarray(Image.open(path), dtype=np.float32) / 255.0
    return array


def letterbox_square(image: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:
    tensor = torch.as_tensor(image, dtype=torch.float32).reshape(1, 1, *image.shape)
    height, width = image.shape
    scale = size / max(height, width)
    new_h, new_w = max(1, int(round(height * scale))), max(1, int(round(width * scale)))
    resized = F.interpolate(tensor, size=(new_h, new_w), mode="bilinear", align_corners=False)[0, 0]
    canvas = torch.zeros(size, size, dtype=torch.float32)
    top, left = (size - new_h) // 2, (size - new_w) // 2
    canvas[top : top + new_h, left : left + new_w] = resized
    return canvas.numpy()


def rotate_np(image: np.ndarray, angle_deg: float) -> np.ndarray:
    pil = Image.fromarray((np.clip(image, 0, 1) * 255).astype(np.uint8))
    rotated = pil.rotate(angle_deg, resample=Image.BILINEAR, fillcolor=0)
    return np.asarray(rotated, dtype=np.float32) / 255.0


def apply_corner_masks(image: np.ndarray, rng: random.Random) -> np.ndarray:
    height, width = image.shape
    out = image.copy()
    corners = [(0, 0), (0, 1), (1, 0), (1, 1)]  # (row_side, col_side): 0=верх/лево, 1=низ/право
    for row_side, col_side in corners:
        if rng.random() >= AUG_MASK_PROB:
            continue
        frac_h = rng.uniform(0.0, AUG_MASK_MAX_FRACTION)
        frac_w = rng.uniform(0.0, AUG_MASK_MAX_FRACTION)
        h = max(1, int(round(height * frac_h)))
        w = max(1, int(round(width * frac_w)))
        r0 = 0 if row_side == 0 else height - h
        c0 = 0 if col_side == 0 else width - w
        out[r0 : r0 + h, c0 : c0 + w] = 0.0
    return out


def to_model_tensor(square: np.ndarray) -> torch.Tensor:
    tensor = torch.as_tensor(square, dtype=torch.float32)
    channels = tensor.expand(3, -1, -1).clone()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (channels - mean) / std


class HipPositioningDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, *, augment: bool, seed: int = SEED):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment
        self.rng = random.Random(seed)
        self.raw = [load_png_float(p) for p in self.frame["png_path"]]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index: int):
        image = self.raw[index]
        label = float(self.frame.loc[index, "label_hip_positioning"])
        if self.augment:
            angle = self.rng.uniform(-AUG_MAX_ROTATION_DEG, AUG_MAX_ROTATION_DEG)
            image = rotate_np(image, angle)
            image = apply_corner_masks(image, self.rng)
            brightness = 1.0 + self.rng.uniform(-AUG_BRIGHTNESS, AUG_BRIGHTNESS)
            contrast = 1.0 + self.rng.uniform(-AUG_CONTRAST, AUG_CONTRAST)
            mean = float(image.mean())
            image = np.clip((image - mean) * contrast + mean * brightness, 0.0, 1.0)
        square = letterbox_square(image)
        return to_model_tensor(square), torch.tensor(label, dtype=torch.float32)

## 5. Модель

In [ ]:
def build_model(pretrained: bool = True) -> nn.Module:
    from torchvision.models import ResNet18_Weights, resnet18
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

## 6. Обучение одного фолда/финальной модели

Взвешенный `BCEWithLogitsLoss` (pos_weight под дисбаланс) + oversampling
позитивов в `WeightedRandomSampler` — обе меры дисбаланса вместе, дисбаланс
сильный (29 позитивов на 122 train).

In [ ]:
EPOCHS = 18
BATCH_SIZE = 16
LEARNING_RATE = 3e-4


def make_loader(frame: pd.DataFrame, *, augment: bool, seed: int) -> DataLoader:
    dataset = HipPositioningDataset(frame, augment=augment, seed=seed)
    if augment:
        labels_arr = frame["label_hip_positioning"].values
        n_pos, n_neg = max(1, labels_arr.sum()), max(1, len(labels_arr) - labels_arr.sum())
        weights = np.where(labels_arr == 1, 1.0 / n_pos, 1.0 / n_neg)
        sampler = torch.utils.data.WeightedRandomSampler(
            weights, num_samples=len(weights), replacement=True,
            generator=torch.Generator().manual_seed(seed),
        )
        return DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)


def train_model(train_frame: pd.DataFrame, *, epochs: int = EPOCHS, seed: int = SEED, verbose: bool = False) -> nn.Module:
    torch.manual_seed(seed)
    loader = make_loader(train_frame, augment=True, seed=seed)

    labels_arr = train_frame["label_hip_positioning"].values
    n_pos, n_neg = max(1, int(labels_arr.sum())), max(1, len(labels_arr) - int(labels_arr.sum()))
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)

    model = build_model(pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(epochs):
        model.train()
        total = 0.0
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images).squeeze(1), targets)
            loss.backward()
            optimizer.step()
            total += float(loss.detach()) * len(targets)
        if verbose and (epoch + 1) % 6 == 0:
            print(f"  эпоха {epoch + 1}/{epochs}: loss={total / len(train_frame):.4f}")
    return model


@torch.no_grad()
def predict_proba(model: nn.Module, frame: pd.DataFrame) -> np.ndarray:
    model.eval()
    loader = make_loader(frame, augment=False, seed=SEED)
    probs = []
    for images, _ in loader:
        probs.extend(torch.sigmoid(model(images.to(DEVICE))[:, 0]).cpu().tolist())
    return np.array(probs)

## 7. 4-фолдовая групповая CV по `study_folder` внутри train -> OOF

In [ ]:
N_FOLDS = 4
cv_t0 = time.time()

group_kfold = GroupKFold(n_splits=N_FOLDS)
oof_prob = np.full(len(train_df), np.nan)

for fold, (fit_idx, hold_idx) in enumerate(
    group_kfold.split(train_df, groups=train_df["study_folder"])
):
    fit_frame = train_df.iloc[fit_idx].reset_index(drop=True)
    hold_frame = train_df.iloc[hold_idx].reset_index(drop=True)
    print(f"фолд {fold + 1}/{N_FOLDS}: fit={len(fit_frame)}, hold={len(hold_frame)}")

    fold_model = train_model(fit_frame, seed=SEED + fold, verbose=False)
    hold_prob = predict_proba(fold_model, hold_frame)
    oof_prob[hold_idx] = hold_prob

assert not np.isnan(oof_prob).any(), "OOF должен покрывать весь train"
cv_elapsed = time.time() - cv_t0
print(f"вложенная CV: {cv_elapsed / 60:.1f} мин")

## 8. Финальная модель на всём train -> предсказания на val

In [ ]:
final_t0 = time.time()
final_model = train_model(train_df, seed=SEED, verbose=True)
val_prob = predict_proba(final_model, val_df)
final_elapsed = time.time() - final_t0
print(f"финальное обучение: {final_elapsed / 60:.1f} мин")

total_elapsed = time.time() - T_START
print(f"итого с начала ноутбука: {total_elapsed / 60:.1f} мин (бюджет {TIME_BUDGET_MIN} мин)")

## 9. Метрики: AUC OOF/val, F1 на пороге max-F1 (OOF)

In [ ]:
y_train = train_df["label_hip_positioning"].values
y_val = val_df["label_hip_positioning"].values

auc_oof = float(roc_auc_score(y_train, oof_prob)) if len(set(y_train)) > 1 else None
auc_val = float(roc_auc_score(y_val, val_prob)) if len(set(y_val)) > 1 else None

thresholds = np.linspace(0.05, 0.95, 91)
f1_scores = [f1_score(y_train, (oof_prob >= t).astype(int)) for t in thresholds]
best_idx = int(np.argmax(f1_scores))
best_threshold = float(thresholds[best_idx])
f1_oof_at_best = float(f1_scores[best_idx])
f1_val_at_oof_threshold = float(f1_score(y_val, (val_prob >= best_threshold).astype(int)))

report = {
    "seed": SEED,
    "n_folds": N_FOLDS,
    "train_n": int(len(train_df)),
    "train_positives": int(y_train.sum()),
    "val_n": int(len(val_df)),
    "val_positives": int(y_val.sum()),
    "auc_oof": auc_oof,
    "auc_val": auc_val,
    "max_f1_threshold_oof": best_threshold,
    "f1_oof_at_max_f1_threshold": f1_oof_at_best,
    "f1_val_at_oof_threshold": f1_val_at_oof_threshold,
    "cv_elapsed_sec": round(cv_elapsed, 1),
    "final_train_elapsed_sec": round(final_elapsed, 1),
    "total_elapsed_sec": round(time.time() - T_START, 1),
    "input_size": INPUT_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
}
print(json.dumps(report, ensure_ascii=False, indent=2))

## 10. Сохранение результатов на Drive

In [ ]:
output_dir = Path(OUTPUT_DIR_DRIVE_PATH)
output_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "dedup_group_id": train_df["dedup_group_id"],
    "prob": oof_prob,
}).to_csv(output_dir / "oof_train.csv", index=False)

pd.DataFrame({
    "dedup_group_id": val_df["dedup_group_id"],
    "prob": val_prob,
}).to_csv(output_dir / "val_pred.csv", index=False)

torch.save(
    {
        "state_dict": final_model.state_dict(),
        "input_size": INPUT_SIZE,
        "normalize_mean": list(IMAGENET_MEAN),
        "normalize_std": list(IMAGENET_STD),
        "seed": SEED,
    },
    output_dir / "hip_positioning_cnn.pt",
)

(output_dir / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("сохранено в", output_dir)
print(sorted(p.name for p in output_dir.iterdir()))

## Что дальше (вручную, не в этом ноутбуке)

Координатор скачивает из `OUTPUT_DIR_DRIVE_PATH`:
* `hip_positioning_cnn.pt` -> `artifacts/models/hip_positioning_cnn.pt` (запись в `WEIGHTS.md`: sha256 + эта команда/ноутбук как источник);
* `report.json`, `oof_train.csv`, `val_pred.csv` -> `artifacts/` для решения, подключать ли CNN-ветку в `hip_checker`
  (сравнение `auc_oof`/`f1_oof_at_max_f1_threshold` с rule-based чекером `hip_positioning`).
  Локальный рантайм (`columba.hip_cnn.load_hip_positioning_predictor`) без файла весов честно возвращает `None` —
  подключение CNN в инференс всегда опционально.